In [ ]:
# Cell 1: High-Stability Installer

!pip install -U pip
!pip install -U unsloth transformers accelerate bitsandbytes xformers
!pip install -q sacrebleu rouge-score
!pip install -q unbabel-comet --no-deps
!pip install -q entmax pytorch-lightning sentencepiece "jsonargparse==3.13.1"

print("✅ Packages installed. Please RESTART the runtime now (Runtime → Restart session).")

In [ ]:
# Cell 2: Imports and Verification 
import os
import re
import time
import torch
import pandas as pd
import numpy as np
import sacrebleu
from unsloth import FastVisionModel
from datasets import load_dataset
from tqdm import tqdm
from rouge_score import rouge_scorer
from comet import download_model, load_from_checkpoint


print(f"✅ NumPy Version: {np.__version__}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")
print(f"✅ Unsloth & Torch Loaded!")
print("✅ Environment ready!")

In [ ]:
# Cell 3: Full Evaluation Script

# --- 1. Load Model ---
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/GLM-4-9B-0414-unsloth-bnb-4bit",   
    max_seq_length = 4096,           
    dtype = None,
    load_in_4bit = True,
    device_map = "cuda:0",           
)
FastVisionModel.for_inference(model)

model = model.to("cuda:0")           

# --- 2. LOAD TEST DATA ---
test_dataset = load_dataset("csv", data_files={"test": "/input/datasets/flores_200_cmn_en.csv"})["test"]
# test_dataset = load_dataset("csv", data_files={"test": "/input/datasets/flores_200_en_cmn.csv"})["test"]
# test_dataset = load_dataset("csv", data_files={"test": "/input/datasets/flores_200_yue_en.csv"})["test"]
# test_dataset = load_dataset("csv", data_files={"test": "/input/datasets/flores_200_en_yue.csv"})["test"]

# --- 3. GENERATE TRANSLATIONS ---
src_list = test_dataset["src"]
ref_list = test_dataset["tgt"]
hyp_list = []

# --- 4. PROMPT TEMPLATE ---
def make_prompt_cmn_en(src_text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": (
                    "You are a professional Mandarin to English translator.\n\n"
                    "TASK: Translate the Mandarin text below into natural, fluent English.\n\n"
                    "CRITICAL RULES:\n"
                    "1. Translate into natural, idiomatic English (not literal word-for-word).\n"
                    "2. Use proper English grammar, spelling, and punctuation.\n"
                    "3. Preserve ALL numbers exactly as they appear in the source.\n"
                    "4. Preserve ALL times exactly (e.g. 11:00, 9:30, UTC+1).\n"
                    "5. Output ONE sentence only. Stop immediately after the translation.\n"
                    "6. No explanations, no notes, no commentary, no repetition.\n\n"
                    f"Mandarin: {src_text}\n\n"
                    "English translation:"
                )}
            ]
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def make_prompt_en_cmn(src_text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": (
                    "You are a professional English to Mandarin translator.\n\n"
                    "TASK: Translate the English text below into natural, fluent Mandarin Chinese (普通话).\n\n"
                    "CRITICAL RULES:\n"
                    "1. Use natural, everyday Mandarin phrasing and vocabulary.\n"
                    "2. Use Simplified Chinese characters (not Traditional).\n"
                    "3. Use ONLY Chinese characters. No Pinyin, no Jyutping, no Latin letters except proper nouns.\n"
                    "4. Preserve ALL numbers exactly as they appear in the source.\n"
                    "5. Preserve ALL times exactly (e.g. 11:00, 9:30, UTC+1).\n"
                    "6. Output ONE sentence only. Stop immediately after the translation.\n"
                    "7. No explanations, no notes, no commentary, no repetition.\n\n"
                    f"English: {src_text}\n\n"
                    "Mandarin translation:"
                )}
            ]
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def make_prompt_en_yue(src_text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": (
                    "You are a professional English to Cantonese translator.\n\n"
                    "TASK: Translate the English text below into natural, colloquial Cantonese (粵語 / 廣東話).\n\n"
                    "CRITICAL RULES:\n"
                    "1. Use natural Cantonese phrasing and vocabulary (e.g., use 喺 instead of 在, 佢 instead of 他, 咁 instead of 了).\n"
                    "2. Use Traditional Chinese characters standard for Cantonese writing. No Simplified Chinese characters.\n"
                    "3. Use ONLY Chinese characters. No Jyutping, no Pinyin, no Latin letters except proper nouns.\n"
                    "4. Preserve ALL numbers exactly as they appear in the source.\n"
                    "5. Preserve ALL times exactly (e.g. 11:00, 9:30, UTC+1).\n"
                    "6. Output ONE sentence only. Stop immediately after the translation.\n"
                    "7. No explanations, no notes, no commentary, no repetition.\n\n"
                    f"English: {src_text}\n\n"
                    "Cantonese translation:"
                )}
            ]
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def make_prompt_yue_en(src_text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": (
                    "You are a professional Cantonese to English translator.\n\n"
                    "TASK: Translate the Cantonese text below into natural, fluent English.\n\n"
                    "CRITICAL RULES:\n"
                    "1. Translate into natural, idiomatic English (not literal word-for-word).\n"
                    "2. Use proper English grammar, spelling, and punctuation.\n"
                    "3. Preserve ALL numbers exactly as they appear in the source.\n"
                    "4. Preserve ALL times exactly (e.g. 11:00, 9:30, UTC+1).\n"
                    "5. Output ONE sentence only. Stop immediately after the translation.\n"
                    "6. No explanations, no notes, no commentary, no repetition.\n\n"
                    f"Cantonese: {src_text}\n\n"
                    "English translation:"
                )}
            ]
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

# --- 5. EXTRACTION (GLM can emit <think>...</think> reasoning, sometimes in Chinese, before the answer) ---

def extract_translation(decoded_text):
    import re
    
    # 1. Remove all GLM special tokens and think blocks
    text = re.sub(r'<\|.*?\|>', '', decoded_text)
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<think>.*', '', text, flags=re.DOTALL)
    text = text.strip()

    if not text:
        return ""

    # 2. Keep only lines that contain at least one Chinese character
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    chinese_lines = [line for line in lines if re.search(r'[\u4e00-\u9fff]', line)]

    if chinese_lines:
        return ' '.join(chinese_lines).strip()

    # 3. Fallback: return the first non-empty line (or the whole text if no line break)
    if lines:
        return lines[0]

    return text


# --- 5. GENERATION LOOP ---
start_time = time.time()
for i, src in enumerate(tqdm(src_list)):
    prompt = make_prompt_cmn_en(src)      # change prompt function for different directions cmn<->en ; yue<->en

    inputs = tokenizer(
        text=[prompt],                    
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=768,
        add_special_tokens=False,
    ).to("cuda:0")

    with torch.no_grad():
        outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        use_cache = True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    hyp_list.append(extract_translation(decoded))

    if (i + 1) % 10 == 0:
        elapsed = time.time() - start_time
        per_sentence = elapsed / (i + 1)
        remaining = per_sentence * (len(src_list) - i - 1)
        print(f"[{i+1}/{len(src_list)}] "
              f"{per_sentence:.1f}s/sentence | "
              f"ETA: {remaining/60:.1f} min")

# --- 6. SAVE BASELINE RESULTS ---
pd.DataFrame({
    "src": src_list,
    "ref": ref_list,
    "hyp": hyp_list
}).to_csv("GLM4_9B_cmn_en_results.csv", index=False)
print("✅ Base results saved to GLM4_9B_cmn_en_results.csv")

In [ ]:
# Cell 4: Full Evaluation Script

# ====================== COMET ======================
print("Loading COMET model...")
model_path = download_model("Unbabel/wmt22-comet-da")

# Optional forward patch (you already saw it worked)
import comet.encoders.xlmr as xlmr_module
if hasattr(xlmr_module, "XLMREncoder"):
    def patched_forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask,
                             output_hidden_states=True, return_dict=True)
        return {
            "last_hidden_state": outputs.last_hidden_state,
            "all_layers": outputs.hidden_states
        }
    xlmr_module.XLMREncoder.forward = patched_forward
    print("✅ Forward patch applied")

comet_model = load_from_checkpoint(model_path)

comet_data = [{"src": s, "mt": h, "ref": r}
              for s, h, r in zip(src_list, hyp_list, ref_list)]

print("Running COMET prediction...")
comet_output = comet_model.predict(
    comet_data,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0,
    progress_bar=True
)

avg_comet = comet_output.system_score
print(f"✨ COMET Score: {avg_comet:.4f}")


# ====================== METRICS ======================
print("\nCalculating BLEU, ChrF...")

bleu = sacrebleu.corpus_bleu(hyp_list, [ref_list], tokenize='zh')
chrf = sacrebleu.corpus_chrf(hyp_list, [ref_list])

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = [scorer.score(r, h) for r, h in zip(ref_list, hyp_list)]

avg_r1 = np.mean([s['rouge1'].fmeasure for s in rouge_scores])
avg_r2 = np.mean([s['rouge2'].fmeasure for s in rouge_scores])
avg_rl = np.mean([s['rougeL'].fmeasure for s in rouge_scores])

print(f"BLEU:  {bleu.score:.2f}")
print(f"ChrF:  {chrf.score:.2f}")
print(f"ROUGE-1: {avg_r1:.4f} | ROUGE-2: {avg_r2:.4f} | ROUGE-L: {avg_rl:.4f}")

# ====================== FINAL REPORT ======================
print("\n" + "="*50)
print("     GLM4_9B cmn_en Final Evaluation")
print("="*50)
print(f"COMET Score:    {avg_comet:.4f}")
print(f"BLEU Score:     {bleu.score:.2f}")
print(f"ChrF Score:     {chrf.score:.2f}")
print(f"ROUGE-1:        {avg_r1:.4f}")
print(f"ROUGE-2:        {avg_r2:.4f}")
print(f"ROUGE-L:        {avg_rl:.4f}")
print("="*50)

# ====================== SAVE METRICS ======================
metrics_df = pd.DataFrame({
    "Metric": ["COMET", "BLEU", "ChrF", "ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Value": [avg_comet, bleu.score, chrf.score, avg_r1, avg_r2, avg_rl]
})
metrics_df.to_csv("GLM4_9B_cmn_en_metrics.csv", index=False)
metrics_df.to_csv("/kaggle/working/GLM4_9B_cmn_en_metrics_report.csv", index=False)
print("✅ Metrics saved to GLM4_9B_cmn_en_metrics.csv")